# Test de normalisation + Q — `TelemacDatasetWithQ`

Ce notebook vérifie la nouvelle classe dataset avec débit global `Q(t)`:
- chargement des `.liq`
- alignement `(x,y,ts)` des `.pkl`
- interpolation de `Q`
- normalisation de `Q`
- cohérence train/eval des stats sauvegardées.


In [2]:
import os
import sys
import shutil
import pickle
import numpy as np

project_path = os.path.abspath(os.path.join(os.getcwd(), '..', ''))
if project_path not in sys.path:
    sys.path.append(project_path)

from python.create_dgl_dataset import (
    TelemacDatasetWithQ,
    unpack_dynamic_sample,
    load_liq_hydrograph,
)


In [6]:
# ====== CONFIG ======
# Adapte ces chemins à ton setup
DATA_DIR = "/work/m24046/m24046mrcr/dataset_x8_avec_ts/shortx8/Mesh8_base.bin"

DYNAMIC_FILES = [
     "/work/m24046/m24046mrcr/dataset_x8_avec_ts/shortx8/Group_1_peak_1000_Group_1_peak_1000_0_35-64_interpolated.pkl",
]

HYDRO_FILES = [
     "/work/m24046/m24046mrcr/dataset_x8_avec_ts/hydrogrammes/generated_hydrographs_Group_1_peak_1000.liq",
]

CKPT_DIR = "./test/ckpt_norm_q_tests"
SEQUENCE_LENGTH = 4
OVERLAP = 3
MAX_SEQ_CHECK = 5
ATOL = 1e-5
RTOL = 1e-5

assert len(DYNAMIC_FILES) > 0, "Renseigne DYNAMIC_FILES"
assert len(DYNAMIC_FILES) == len(HYDRO_FILES), "DYNAMIC_FILES et HYDRO_FILES doivent être alignées"


In [7]:
# reset stats dir
if os.path.isdir(CKPT_DIR):
    shutil.rmtree(CKPT_DIR)
os.makedirs(CKPT_DIR, exist_ok=True)
print('CKPT_DIR ready:', CKPT_DIR)


CKPT_DIR ready: ./test/ckpt_norm_q_tests


## 1) Test parser `.liq` et format `(x,y,ts)` des `.pkl`

In [8]:
for hp in HYDRO_FILES:
    t_arr, q_arr = load_liq_hydrograph(hp)
    assert len(t_arr) > 1, f"Hydro vide/invalide: {hp}"
    assert np.all(np.diff(t_arr) >= 0), f"Temps non monotones: {hp}"
    assert np.all(np.isfinite(q_arr)), f"Q non fini: {hp}"
print('OK parser .liq')

for pkl_path in DYNAMIC_FILES:
    with open(pkl_path, 'rb') as f:
        data = pickle.load(f)
    assert len(data) > 0, f"PKL vide: {pkl_path}"
    x, y, ts = unpack_dynamic_sample(data[0])
    assert ts is not None, f"ts manquant dans {pkl_path}"
    assert x.shape[0] == y.shape[0], f"x/y mismatch dans {pkl_path}"
print('OK format pkl avec ts')


OK parser .liq
OK format pkl avec ts


## 2) Chargement `train` (calcule + sauvegarde des stats, y compris `q` )

In [9]:
train_set = TelemacDatasetWithQ(
    name='telemac_train_q',
    data_dir=DATA_DIR,
    dynamic_data_files=DYNAMIC_FILES,
    hydro_data_files=HYDRO_FILES,
    split='train',
    ckpt_path=CKPT_DIR,
    normalize=True,
    sequence_length=SEQUENCE_LENGTH,
    overlap=OVERLAP,
    dt_seconds=1800.0,
)

print('Nombre de séquences train:', len(train_set))
print('Clés node_stats:', sorted(list(train_set.node_stats.keys())))
assert 'q' in train_set.node_stats and 'q_std' in train_set.node_stats, 'q stats manquantes'


Normalizing data...
Nombre de séquences train: 27
Clés node_stats: ['delta_h', 'delta_h_std', 'delta_u', 'delta_u_std', 'delta_v', 'delta_v_std', 'h', 'h_std', 'q', 'q_std', 'strickler', 'strickler_std', 'u', 'u_std', 'v', 'v_std', 'z', 'z_std']


In [10]:
dyn_off = train_set.base_graph.ndata['static'].shape[1]
print('dyn_off=', dyn_off, '(attendu 6 pour static)')

def expected_q_norm(ds, seq_idx, t_idx):
    sample = ds.sequences[seq_idx][t_idx]
    _, _, ts = unpack_dynamic_sample(sample)
    hydro_path = ds.sequence_meta[seq_idx]['hydro_path']

    # interpolation indépendante (sans _q_at_ts)
    t_arr, q_arr = ds.hydrographs[hydro_path]
    q_raw = float(np.interp(float(ts) * ds.dt_seconds, t_arr, q_arr, left=q_arr[0], right=q_arr[-1]))

    q_mean = float(ds.node_stats['q'])
    q_std = float(ds.node_stats['q_std'])
    if q_std != 0.0:
        return (q_raw - q_mean) / q_std
    return q_raw


dyn_off= 6 (attendu 6 pour static)


## 3) Vérifications Q dans les graphes du dataset

In [11]:
n_seq = min(MAX_SEQ_CHECK, len(train_set))
for seq_idx in range(n_seq):
    graphs = train_set[seq_idx]
    for t_idx, g in enumerate(graphs):
        x = g.ndata['x'].cpu().numpy()
        dyn = x[:, dyn_off:dyn_off+4]  # [h,u,v,q]
        q_col = dyn[:, 3]

        # Q broadcast (constant spatialement)
        assert np.allclose(q_col, q_col[0], atol=ATOL, rtol=RTOL), (
            f'Q non constant sur les noeuds (seq={seq_idx}, t={t_idx})'
        )

        # Q attendu par interpolation + normalisation
        q_exp = expected_q_norm(train_set, seq_idx, t_idx)
        assert np.allclose(q_col[0], q_exp, atol=ATOL, rtol=RTOL), (
            f'Mismatch Q norm (seq={seq_idx}, t={t_idx}) got={q_col[0]} exp={q_exp}'
        )

print('OK: broadcast Q + interpolation + normalisation sur train_set')
print('q_mean=', float(train_set.node_stats['q']), 'q_std=', float(train_set.node_stats['q_std']))


OK: broadcast Q + interpolation + normalisation sur train_set
q_mean= 790.2642822265625 q_std= 133.81561279296875


## 4) Chargement `eval` (recharge stats train)

In [12]:
eval_set = TelemacDatasetWithQ(
    name='telemac_eval_q',
    data_dir=DATA_DIR,
    dynamic_data_files=DYNAMIC_FILES,
    hydro_data_files=HYDRO_FILES,
    split='eval',
    ckpt_path=CKPT_DIR,
    normalize=True,
    sequence_length=SEQUENCE_LENGTH,
    overlap=OVERLAP,
    dt_seconds=1800.0,
)

assert np.isclose(float(train_set.node_stats['q']), float(eval_set.node_stats['q']))
assert np.isclose(float(train_set.node_stats['q_std']), float(eval_set.node_stats['q_std']))
print('OK: q/q_std cohérents train vs eval')


Loading normalization statistics...
OK: q/q_std cohérents train vs eval


## 5) Round-trip rapide sur Q normalisé

In [13]:
g0 = train_set[0][0]
x0 = g0.ndata['x'][:, dyn_off:dyn_off+4].cpu().numpy()
q_norm = x0[:10, 3:4]

q_mean = float(train_set.node_stats['q'])
q_std = float(train_set.node_stats['q_std'])

q_denorm = q_norm * q_std + q_mean if q_std != 0.0 else q_norm
q_renorm = (q_denorm - q_mean) / q_std if q_std != 0.0 else q_denorm

assert np.allclose(q_norm, q_renorm, atol=ATOL, rtol=RTOL)
print('OK round-trip Q')


OK round-trip Q


✅ Tous les tests principaux sont passés si aucune assertion n'a échoué.